# Notebook 1: Vision Transfer Learning — CNN vs ViT vs DINOv2 on EuroSAT

This notebook provides a comprehensive comparison of four vision model families for satellite image classification:
- **EfficientNet-B0** — efficient CNN scaling (Tan & Le, 2019)
- **ResNet-50** — classic residual CNN baseline (He et al., 2016)
- **ViT-Base/16** — pure transformer vision model (Dosovitskiy et al., 2021)
- **DINOv2-Base** — self-supervised ViT with strong transfer properties (Oquab et al., 2023)

**Key questions answered:**
1. Which model achieves the best accuracy on EuroSAT with full fine-tuning?
2. How do the three adaptation strategies (linear probe, partial unfreeze, full fine-tune) compare?
3. How data-efficient is each model? Does DINOv2's self-supervised pretraining help with fewer labels?
4. What is the latency–accuracy trade-off for production deployment?
5. What do ViT/DINOv2 attention maps reveal about their representations?

## Section 0: Imports & Environment Setup

In [ ]:
import sys; sys.path.insert(0, '..')
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import seaborn as sns
import torch
from datasets import load_dataset
from PIL import Image

from configs.vision_config import (
    DATA_FRACTIONS,
    EUROSAT_CLASSES,
    STRATEGIES,
    VISION_MODELS,
    VisionTrainingConfig,
)
from src.utils.metrics import (
    benchmark_latency,
    benchmark_onnx_latency,
    export_to_onnx,
)
from src.utils.visualization import (
    compute_attention_rollout,
    plot_attention_rollout,
    plot_confusion_matrix,
    plot_data_efficiency_curves,
    plot_latency_accuracy_scatter,
)
from src.vision.model import build_model, count_trainable_params
from src.vision.trainer import train_vision_model

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

## Section 1: Dataset Exploration — EuroSAT

**EuroSAT** (Helber et al., 2019) is a land-use classification dataset derived from Sentinel-2 satellite imagery. It contains **27,000 labeled patches** across **10 land-use classes** at 64×64 pixels.

### Why EuroSAT is a strong benchmark for transfer learning:
- **Large domain gap from ImageNet**: Satellite imagery is top-down, multispectral, and visually very different from natural photographs that ImageNet pretrained models were trained on. This means naive fine-tuning can easily fail.
- **Class imbalance is minimal**: EuroSAT is relatively balanced (~2,700 images/class), allowing fair comparison.
- **Diverse semantics**: Classes range from fine-grained (e.g., Highway vs Industrial) to visually distinct (Forest vs Sea/Lake), testing both coarse and fine-grained discrimination.
- **Practical relevance**: Remote sensing is a high-impact application domain — disaster response, urban planning, agricultural monitoring all rely on it.

The 10 classes are: `AnnualCrop`, `Forest`, `HerbaceousVegetation`, `Highway`, `Industrial`, `Pasture`, `PermanentCrop`, `Residential`, `River`, `SeaLake`.

In [ ]:
# Load EuroSAT from HuggingFace Datasets
dataset = load_dataset("blanchon/EuroSAT_RGB")
train_ds = dataset["train"]
test_ds  = dataset["test"] if "test" in dataset else dataset["validation"]

print(f"Train size: {len(train_ds):,}  |  Test size: {len(test_ds):,}")
print(f"Classes ({len(EUROSAT_CLASSES)}): {EUROSAT_CLASSES}")

# Class distribution bar chart
from collections import Counter

train_labels = train_ds["label"]
counts = Counter(train_labels)
sorted_counts = [(EUROSAT_CLASSES[k], counts[k]) for k in sorted(counts)]

fig, ax = plt.subplots(figsize=(10, 4))
classes_sorted, freqs = zip(*sorted_counts)
bars = ax.bar(classes_sorted, freqs, color=sns.color_palette("viridis", len(classes_sorted)))
ax.set_title("EuroSAT Training Set — Class Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Land-Use Class")
ax.set_ylabel("Number of Images")
ax.tick_params(axis="x", rotation=35)
for bar, count in zip(bars, freqs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20, str(count),
            ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig("../results/figures/eurosat_class_distribution.png", dpi=150)
plt.show()

In [ ]:
# Display a 3×3 grid of sample images (3 classes × 3 examples each)
target_classes = [0, 1, 4]  # AnnualCrop, Forest, Industrial
samples_per_class = 3

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
fig.suptitle("EuroSAT Sample Images — 3 Classes × 3 Samples", fontsize=14, fontweight="bold")

for row_idx, cls_idx in enumerate(target_classes):
    cls_samples = [x for x in train_ds if x["label"] == cls_idx][:samples_per_class]
    for col_idx, sample in enumerate(cls_samples):
        ax = axes[row_idx][col_idx]
        img = sample["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.array(img))
        ax.imshow(img)
        ax.axis("off")
        if col_idx == 0:
            ax.set_ylabel(EUROSAT_CLASSES[cls_idx], fontsize=10, rotation=0,
                          labelpad=60, va="center", fontweight="bold")
        if row_idx == 0:
            ax.set_title(f"Sample {col_idx + 1}", fontsize=9)

plt.tight_layout()
plt.savefig("../results/figures/eurosat_sample_grid.png", dpi=150)
plt.show()

## Section 2: Model Architecture Overview

We evaluate four pretrained vision backbones spanning three architectural paradigms:

| Model | Year | Params (M) | Architecture | Pretraining | Notes |
|-------|------|-----------|-------------|-------------|-------|
| **EfficientNet-B0** | 2019 | 5.3M | CNN (compound scaling) | ImageNet-1K supervised | Excellent efficiency–accuracy trade-off; strong baseline |
| **ResNet-50** | 2016 | 25.6M | CNN (residual blocks) | ImageNet-1K supervised | Classic baseline; widely used in remote sensing |
| **ViT-Base/16** | 2021 | 86M | Pure Transformer (16×16 patches) | ImageNet-21K supervised | Requires large data; global attention captures spatial structure |
| **DINOv2-Base** | 2023 | 86M | ViT-Base + DINOv2 SSL | LVD-142M self-supervised | Strong zero-shot/linear probe; no task-specific pretraining |

### Three Adaptation Strategies:
1. **Linear Probe**: Freeze all backbone weights; only train a new classification head. Tests raw representation quality.
2. **Partial Unfreeze**: Unfreeze the last N layers of the backbone + classification head. Balances speed and adaptation.
3. **Full Fine-Tune**: Train all weights end-to-end. Maximum capacity but risk of catastrophic forgetting on small datasets.

In [ ]:
# Load all 4 models and print trainable parameter counts per strategy
print(f"{'Model':<20} {'Strategy':<20} {'Trainable Params':>20} {'% of Total':>12}")
print("-" * 76)

param_table = {}

for model_key in VISION_MODELS:
    param_table[model_key] = {}
    for strategy in STRATEGIES:
        cfg = VisionTrainingConfig(
            model_key=model_key,
            strategy=strategy,
            data_fraction=1.0,
            num_epochs=1,  # just for loading
        )
        model = build_model(cfg)
        trainable, total = count_trainable_params(model)
        pct = 100.0 * trainable / total
        param_table[model_key][strategy] = {"trainable": trainable, "total": total, "pct": pct}
        print(f"{model_key:<20} {strategy:<20} {trainable:>20,} {pct:>11.1f}%")
    print()

## Section 3: Strategy Comparison — Full Data

### The three transfer learning strategies in depth:

**Linear Probe** is the gold standard for measuring representation quality. If a linear classifier trained on frozen features achieves high accuracy, it means the pretrained representations already encode task-relevant information. DINOv2 was specifically designed to produce such linearly separable features via its distillation + self-supervised objective.

**Partial Unfreeze** (also called "staged fine-tuning") unfreezes the top layers of the network. This exploits the well-known layer hierarchy in deep networks: early layers capture universal features (edges, textures) while later layers capture task-specific features. For domain-shifted tasks like satellite imagery, the later layers need more adaptation.

**Full Fine-Tune** gives the model maximum flexibility to adapt all representations to EuroSAT. This generally achieves the best results but: (1) requires more data to avoid overfitting, (2) risks catastrophic forgetting, and (3) has the highest computational cost. Using a lower learning rate for the backbone (vs. the head) is standard practice.

In [ ]:
import json

RESULTS_PATH = Path("../results/vision")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

strategy_results_file = RESULTS_PATH / "strategy_results.json"

if strategy_results_file.exists():
    print("Loading pre-computed strategy results...")
    with open(strategy_results_file) as f:
        strategy_results = json.load(f)
else:
    print("Training all 4 models × 3 strategies at 100% data...")
    strategy_results = {}
    for model_key in VISION_MODELS:
        strategy_results[model_key] = {}
        for strategy in STRATEGIES:
            print(f"  Training {model_key} | {strategy}...")
            cfg = VisionTrainingConfig(
                model_key=model_key,
                strategy=strategy,
                data_fraction=1.0,
            )
            metrics = train_vision_model(cfg)
            strategy_results[model_key][strategy] = metrics
            print(f"    → test_acc={metrics['test_acc']:.4f}")
    with open(strategy_results_file, "w") as f:
        json.dump(strategy_results, f, indent=2)
    print("Results saved.")

In [ ]:
# Grouped bar chart: test accuracy across strategies per model
models = list(VISION_MODELS.keys())
strategies = list(STRATEGIES)
colors = sns.color_palette("Set2", len(strategies))

x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, (strategy, color) in enumerate(zip(strategies, colors)):
    accs = [strategy_results[m][strategy]["test_acc"] * 100 for m in models]
    offset = (i - 1) * width
    bars = ax.bar(x + offset, accs, width, label=strategy.replace("_", " ").title(), color=color, edgecolor="white")
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{acc:.1f}", ha="center", va="bottom", fontsize=7, rotation=45)

ax.set_xticks(x)
ax.set_xticklabels([VISION_MODELS[m]["display_name"] for m in models], fontsize=11)
ax.set_ylabel("Test Accuracy (%)", fontsize=12)
ax.set_title("Strategy Comparison — EuroSAT Test Accuracy (Full Data)", fontsize=14, fontweight="bold")
ax.legend(title="Strategy", fontsize=10)
ax.set_ylim(50, 100)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("../results/figures/strategy_comparison_bar.png", dpi=150)
plt.show()

# Summary table
print(f"\n{'Model':<20} ", end="")
for s in strategies:
    print(f"{s.replace('_', ' ').title():<22}", end="")
print()
print("-" * (20 + 22 * len(strategies)))
for m in models:
    print(f"{m:<20} ", end="")
    for s in strategies:
        acc = strategy_results[m][s]["test_acc"] * 100
        print(f"{acc:<22.2f}", end="")
    print()

## Section 4: Data Efficiency Study

### How many labeled examples does each model actually need?

One of the most important practical questions in transfer learning is **label efficiency**: given a limited annotation budget, which pretrained model gets you the best performance?

This matters enormously in remote sensing, where labeling satellite imagery requires expert annotators. Even a 10× reduction in required labels can make a project feasible.

**Hypothesis**: DINOv2's self-supervised pretraining on diverse, large-scale data (LVD-142M) should produce richer, more generalizable representations than supervised ImageNet pretraining — especially at low label counts. We test this hypothesis systematically.

We run full fine-tuning at data fractions: **1%, 5%, 10%, 100%** of the training set.

In [ ]:
data_eff_file = RESULTS_PATH / "data_efficiency_results.json"

if data_eff_file.exists():
    print("Loading pre-computed data efficiency results...")
    with open(data_eff_file) as f:
        data_eff_results = json.load(f)
else:
    print("Running data efficiency study (full_finetune × 4 fractions × 4 models)...")
    data_eff_results = {}
    for model_key in VISION_MODELS:
        data_eff_results[model_key] = {}
        for frac in DATA_FRACTIONS:
            print(f"  {model_key} | {frac*100:.0f}% data...")
            cfg = VisionTrainingConfig(
                model_key=model_key,
                strategy="full_finetune",
                data_fraction=frac,
            )
            metrics = train_vision_model(cfg)
            data_eff_results[model_key][str(frac)] = metrics
            print(f"    → test_acc={metrics['test_acc']:.4f}")
    with open(data_eff_file, "w") as f:
        json.dump(data_eff_results, f, indent=2)
    print("Results saved.")

In [ ]:
# Data efficiency curves
plot_data_efficiency_curves(
    results=data_eff_results,
    data_fractions=DATA_FRACTIONS,
    model_names=VISION_MODELS,
    save_path="../results/figures/data_efficiency_curves.png"
)

# Find crossover point: when does DINOv2 overtake EfficientNet?
fracs = DATA_FRACTIONS
dinov2_accs = [data_eff_results["dinov2"][str(f)]["test_acc"] for f in fracs]
effnet_accs = [data_eff_results["efficientnet"][str(f)]["test_acc"] for f in fracs]

print("\nData Efficiency Summary — Full Fine-Tune")
print(f"{'Fraction':<10} ", end="")
for m in VISION_MODELS:
    print(f"{m:<16}", end="")
print()
print("-" * (10 + 16 * len(VISION_MODELS)))

for frac in fracs:
    n_samples = int(frac * len(train_ds))
    print(f"{frac*100:>5.0f}% ({n_samples:>4}) ", end="")
    for m in VISION_MODELS:
        acc = data_eff_results[m][str(frac)]["test_acc"] * 100
        print(f"{acc:<16.2f}", end="")
    print()

# Identify crossover
crossover_frac = None
for frac, d_acc, e_acc in zip(fracs, dinov2_accs, effnet_accs):
    if d_acc >= e_acc:
        crossover_frac = frac
        break

if crossover_frac:
    print(f"\nKey Finding: DINOv2 overtakes EfficientNet at {crossover_frac*100:.0f}% of training data.")
else:
    print("\nKey Finding: EfficientNet leads across all tested fractions.")

## Section 5: Latency vs Accuracy Analysis

### The production deployment question: accuracy is only half the story.

In real-world remote sensing pipelines, models must process satellite tiles in near-real-time. A model that is 2% more accurate but 10× slower may not be deployable. This section benchmarks:

1. **CPU inference latency** (median over 100 runs) — important for edge/IoT deployment
2. **ONNX-optimized latency** — production standard for CPU serving
3. **Model size** (number of parameters) — affects memory footprint

We plot a **bubble chart** where:
- X-axis = median latency (ms)
- Y-axis = test accuracy (%)
- Bubble size = number of parameters
- Color = model family

The ideal model sits in the **top-left corner** (high accuracy, low latency).

In [ ]:
latency_file = RESULTS_PATH / "latency_results.json"

if latency_file.exists():
    with open(latency_file) as f:
        latency_results = json.load(f)
else:
    latency_results = {}
    onnx_dir = Path("../results/onnx")
    onnx_dir.mkdir(parents=True, exist_ok=True)

    for model_key in VISION_MODELS:
        print(f"Benchmarking {model_key}...")
        cfg = VisionTrainingConfig(model_key=model_key, strategy="full_finetune", data_fraction=1.0)
        model = build_model(cfg)

        # Load best checkpoint
        ckpt_path = RESULTS_PATH / model_key / "full_finetune" / "best_model.pt"
        if ckpt_path.exists():
            model.load_state_dict(torch.load(ckpt_path, map_location="cpu"))
        model.eval()

        # CPU latency
        latency_stats = benchmark_latency(model, input_size=(1, 3, 224, 224), device="cpu", n_runs=100)

        # ONNX export + benchmark
        onnx_path = onnx_dir / f"{model_key}.onnx"
        export_to_onnx(model, onnx_path, input_size=(1, 3, 224, 224))
        onnx_latency = benchmark_onnx_latency(onnx_path, input_size=(1, 3, 224, 224), n_runs=100)

        trainable, total = count_trainable_params(build_model(VisionTrainingConfig(model_key=model_key, strategy="full_finetune")))
        acc = strategy_results[model_key]["full_finetune"]["test_acc"]

        latency_results[model_key] = {
            "cpu_median_ms": latency_stats["median_ms"],
            "cpu_p95_ms": latency_stats["p95_ms"],
            "onnx_median_ms": onnx_latency["median_ms"],
            "test_acc": acc,
            "total_params": total,
        }
        print(f"  CPU: {latency_stats['median_ms']:.1f}ms | ONNX: {onnx_latency['median_ms']:.1f}ms | acc={acc:.4f}")

    with open(latency_file, "w") as f:
        json.dump(latency_results, f, indent=2)

# Latency vs Accuracy scatter
plot_latency_accuracy_scatter(
    latency_results=latency_results,
    model_names=VISION_MODELS,
    save_path="../results/figures/latency_accuracy_scatter.png"
)

# Summary table
print(f"\n{'Model':<20} {'Acc (%)':<10} {'CPU (ms)':<12} {'ONNX (ms)':<12} {'Params (M)':<12}")
print("-" * 68)
for m, stats in latency_results.items():
    print(f"{m:<20} {stats['test_acc']*100:<10.2f} {stats['cpu_median_ms']:<12.1f} {stats['onnx_median_ms']:<12.1f} {stats['total_params']/1e6:<12.1f}")

## Section 6: ViT/DINOv2 Attention Rollout Visualization

### Attention Rollout vs Grad-CAM — which do we prefer and why?

For CNN models (ResNet, EfficientNet), **Grad-CAM** (Selvaraju et al., 2017) is the standard technique for visualizing class activation maps. It uses gradients of the classification score w.r.t. the final convolutional feature map.

For Transformer models (ViT, DINOv2), **Attention Rollout** (Abnar & Zuidema, 2020) is more faithful:
- It recursively multiplies attention weight matrices across all layers, tracking how information flows from input patches to the [CLS] token.
- Unlike raw attention in the last layer (which can be noisy), rollout accounts for the full depth of the model.
- It produces a spatial attention map over the 14×14 patch grid (for ViT-Base/16 on 224×224 images).

**What to look for**: Does the model attend to semantically meaningful regions? On `River` images, does the attention focus on the water channel? On `Industrial`, does it highlight buildings and structures?

In [ ]:
import torchvision.transforms as T

# Load best ViT model
vit_cfg = VisionTrainingConfig(model_key="vit", strategy="full_finetune", data_fraction=1.0)
vit_model = build_model(vit_cfg)
vit_ckpt = RESULTS_PATH / "vit" / "full_finetune" / "best_model.pt"
if vit_ckpt.exists():
    vit_model.load_state_dict(torch.load(vit_ckpt, map_location="cpu"))
vit_model.eval()

# Load best DINOv2 model
dino_cfg = VisionTrainingConfig(model_key="dinov2", strategy="full_finetune", data_fraction=1.0)
dino_model = build_model(dino_cfg)
dino_ckpt = RESULTS_PATH / "dinov2" / "full_finetune" / "best_model.pt"
if dino_ckpt.exists():
    dino_model.load_state_dict(torch.load(dino_ckpt, map_location="cpu"))
dino_model.eval()

print("ViT and DINOv2 models loaded for attention visualization.")

In [ ]:
# Select 6 test images: 3 correct predictions + 3 incorrect predictions
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Collect predictions on test set
correct_samples = []
wrong_samples = []

for sample in test_ds:
    img = sample["image"]
    if not isinstance(img, Image.Image):
        img = Image.fromarray(np.array(img))
    img_rgb = img.convert("RGB")
    tensor = transform(img_rgb).unsqueeze(0)

    with torch.no_grad():
        out = dino_model(tensor)
        logits = out.logits if hasattr(out, "logits") else out
        pred = logits.argmax(dim=-1).item()

    true_label = sample["label"]
    if pred == true_label and len(correct_samples) < 3:
        correct_samples.append({"image": img_rgb, "tensor": tensor, "true": true_label, "pred": pred})
    elif pred != true_label and len(wrong_samples) < 3:
        wrong_samples.append({"image": img_rgb, "tensor": tensor, "true": true_label, "pred": pred})

    if len(correct_samples) == 3 and len(wrong_samples) == 3:
        break

all_samples = correct_samples + wrong_samples
labels = ["Correct"] * 3 + ["Wrong"] * 3
print(f"Selected {len(correct_samples)} correct + {len(wrong_samples)} wrong predictions for visualization.")

In [ ]:
# Compute attention rollout and visualize
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
fig.suptitle("DINOv2 Attention Rollout — Correct (left) vs Incorrect (right) Predictions",
             fontsize=13, fontweight="bold")

col_titles = ["Image", "Attention Map"] * 3

for idx, (sample, result_label) in enumerate(zip(all_samples, labels)):
    col_start = (idx % 3) * 2 if False else idx * 1  # one column per sample for simplicity
    row = idx % 3
    col = (idx // 3) * 3

    rollout = compute_attention_rollout(dino_model, sample["tensor"])

    true_cls = EUROSAT_CLASSES[sample["true"]]
    pred_cls = EUROSAT_CLASSES[sample["pred"]]
    title = f"{result_label}\nTrue: {true_cls}\nPred: {pred_cls}"

    plot_attention_rollout(
        image=sample["image"],
        rollout=rollout,
        ax_img=axes[row][col],
        ax_attn=axes[row][col + 1],
        title=title,
    )

plt.tight_layout()
plt.savefig("../results/figures/attention_rollout.png", dpi=150)
plt.show()

## Section 7: Confusion Matrix + Error Analysis

A confusion matrix at scale reveals systematic failure modes. We expect certain class pairs to be difficult:
- **Highway vs River**: Both are elongated linear structures
- **AnnualCrop vs PermanentCrop**: Similar texture; seasonal variation is the key discriminator
- **Industrial vs Residential**: Both have buildings; scale and density differ

In [ ]:
# Compute full test set predictions with best model (DINOv2 full_finetune)
all_preds = []
all_trues = []

dino_model.eval()
batch_size = 64

for i in range(0, len(test_ds), batch_size):
    batch = test_ds[i:i + batch_size]
    tensors = []
    for img, label in zip(batch["image"], batch["label"]):
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.array(img))
        tensors.append(transform(img.convert("RGB")))
        all_trues.append(label)
    batch_tensor = torch.stack(tensors)
    with torch.no_grad():
        out = dino_model(batch_tensor)
        logits = out.logits if hasattr(out, "logits") else out
        preds = logits.argmax(dim=-1).tolist()
    all_preds.extend(preds)

all_preds = np.array(all_preds)
all_trues = np.array(all_trues)
print(f"Computed predictions for {len(all_preds)} test samples.")
print(f"Overall accuracy: {(all_preds == all_trues).mean()*100:.2f}%")

In [ ]:
# Normalized confusion matrix
plot_confusion_matrix(
    y_true=all_trues,
    y_pred=all_preds,
    class_names=EUROSAT_CLASSES,
    normalize=True,
    title="DINOv2 Full Fine-Tune — EuroSAT Test Set Confusion Matrix",
    save_path="../results/figures/confusion_matrix.png"
)

In [ ]:
# Top-3 most confused class pairs
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_trues, all_preds)
np.fill_diagonal(cm, 0)  # remove correct predictions

confused_pairs = []
for i in range(len(EUROSAT_CLASSES)):
    for j in range(len(EUROSAT_CLASSES)):
        if i != j and cm[i, j] > 0:
            confused_pairs.append((cm[i, j], EUROSAT_CLASSES[i], EUROSAT_CLASSES[j]))

confused_pairs.sort(reverse=True)
print("Top-3 Most Confused Class Pairs:")
print(f"{'Rank':<6} {'True Class':<25} {'Predicted As':<25} {'Count':>8}")
print("-" * 68)
for rank, (count, true_cls, pred_cls) in enumerate(confused_pairs[:3], 1):
    print(f"{rank:<6} {true_cls:<25} {pred_cls:<25} {count:>8}")

print("\nAnalysis:")
for count, true_cls, pred_cls in confused_pairs[:3]:
    print(f"  • {true_cls} → {pred_cls}: {count} misclassifications")

## Section 8: MLflow Experiment Summary

All training runs are logged to MLflow for reproducibility and experiment tracking. Here we load all runs and produce a consolidated comparison.

In [ ]:
import pandas as pd

# Set MLflow tracking URI (adjust if using remote tracking server)
mlflow.set_tracking_uri("../mlruns")

try:
    experiment = mlflow.get_experiment_by_name("vision_transfer_learning")
    if experiment is None:
        print("No MLflow experiment found. Run training first.")
    else:
        runs_df = mlflow.search_runs(
            experiment_ids=[experiment.experiment_id],
            order_by=["metrics.test_acc DESC"]
        )

        cols = ["tags.model_key", "tags.strategy", "params.data_fraction",
                "metrics.test_acc", "metrics.val_acc", "metrics.train_time_s"]
        cols = [c for c in cols if c in runs_df.columns]
        display_df = runs_df[cols].copy()
        display_df.columns = [c.split(".")[-1] for c in cols]
        display_df["test_acc"] = (display_df["test_acc"] * 100).round(2)
        display_df["val_acc"] = (display_df["val_acc"] * 100).round(2)

        print("All MLflow Runs — Vision Transfer Learning Experiment")
        print(display_df.to_string(index=False))

        print("\nBest Model Per Strategy:")
        for strategy in STRATEGIES:
            subset = display_df[display_df["strategy"] == strategy]
            if len(subset) > 0:
                best = subset.loc[subset["test_acc"].idxmax()]
                print(f"  {strategy:<20}: {best['model_key']:<15} → {best['test_acc']:.2f}%")
except Exception as e:
    print(f"MLflow retrieval skipped: {e}")
    print("Summary from saved results:")
    rows = []
    for m in VISION_MODELS:
        for s in STRATEGIES:
            if m in strategy_results and s in strategy_results[m]:
                rows.append({"model": m, "strategy": s, "test_acc": strategy_results[m][s]["test_acc"]*100})
    df = pd.DataFrame(rows)
    print(df.sort_values("test_acc", ascending=False).to_string(index=False))